# Challenger big-trade study

**Hypothesis:** challenger markets are traded by fast, informed funds we can't out-model or out-speed. So instead of predicting, *ride the informed flow* — when a large aggressive trade hits, follow the taker's side.

**Signal:** a trade whose dollar notional (`size × price_paid`) ≥ a threshold.

**Metrics:**
1. *Short-horizon follow-through* — does the YES price keep moving in the trade's direction over the next 1/5/15 min?
2. *Follow-to-settlement PnL* — enter the aggressor's side, hold to settlement (0 or 1), net of Kalshi taker fee.

**Key control:** compare BIG trades vs ALL trades *within price buckets*. Raw win-rates are dominated by a favorite/price confound (big trades cluster on favorites, favorites win) and by path bias (a market heading YES spends lots of time at high yes-prices). The within-bucket comparison is what isolates whether *size* adds information beyond the price.

**Caveats to remember when reading PnL:**
- Entry price is a proxy = the big trade's own fill price. In reality the trade lifts the offer, so following fills you *worse*. Treat PnL as optimistic.
- Data is whatever settled matches you pull — one day is small and correlated. Scale `N_EVENTS` / date range before trusting thin cells (esp. underdog buckets).

## Config — change these freely

In [6]:
BASE      = 'https://external-api.kalshi.com'
SERIES    = 'KXATPCHALLENGERMATCH'   # try 'KXATPMATCH' for main-tour, 'KXWTA...' etc.
STATUS    = 'settled'                # only settled markets have a result to score against
N_EVENTS  = 100                       # how many recent settled events to pull (2 markets each)

BAND        = (0.05, 0.95)           # ignore near-settled prices where the outcome is ~decided
THRESHOLDS  = [0, 100, 250, 500, 1000, 1500, 2000]   # dollar notional; 0 = all-trades baseline
HORIZONS    = [60, 300, 900]         # follow-through horizons, seconds
PRICE_BUCKETS = [(0.05,0.35),(0.35,0.50),(0.50,0.65),(0.65,0.80),(0.80,0.95)]

CACHE_CSV = 'challenger_bigtrade_records.csv'  # raw per-trade records; reused unless REFETCH
REFETCH   = True                    # set True to re-pull from the API (slow)

## Helpers — API + trade tape

In [7]:
import requests, time, os
import bisect
from datetime import datetime
import pandas as pd


def get(path, **params):
    """GET with basic 429 retry."""
    for _ in range(4):
        r = requests.get(BASE + path, params=params, timeout=20)
        if r.status_code == 429:
            time.sleep(1.0); continue
        r.raise_for_status()
        return r.json()
    r.raise_for_status()


def parse_ts(s):
    return datetime.fromisoformat(s.replace('Z', '+00:00')).timestamp()


def fetch_trades(ticker):
    """Full trade tape, ascending by time. Rows: (t, size, side, yes_price, price_paid).
    side = taker_outcome_side ('yes'/'no'); price_paid = what the aggressor paid on that side."""
    out, cursor = [], None
    while True:
        j = get('/trade-api/v2/markets/trades', ticker=ticker, limit=1000,
                **({'cursor': cursor} if cursor else {}))
        for x in j.get('trades', []):
            yp = float(x['yes_price_dollars'])
            side = x['taker_outcome_side']
            price_paid = yp if side == 'yes' else (1 - yp)
            out.append((parse_ts(x['created_time']), float(x['count_fp']), side, yp, price_paid))
        cursor = j.get('cursor')
        if not cursor or not j.get('trades'):
            break
    out.sort(key=lambda r: r[0])
    return out


def yes_price_at(trades, t):
    """Last YES price at or before time t (trades sorted ascending)."""
    ts = [r[0] for r in trades]
    i = bisect.bisect_right(ts, t) - 1
    return trades[i][3] if i >= 0 else None


def taker_fee(price):
    """Kalshi taker fee per contract (dollars): 0.07 * P * (1-P)."""
    return 0.07 * price * (1 - price)

## Build the per-trade record set

One row per in-band trade, tagged with its notional, the settlement outcome of following the aggressor, and the signed price follow-through at each horizon. Cached to CSV — set `REFETCH = True` in the config cell to re-pull.

In [8]:
def build_records():
    evs = get('/trade-api/v2/events', series_ticker=SERIES, status=STATUS, limit=N_EVENTS)['events']
    records, match_summ = [], []
    for ev in evs:
        et = ev['event_ticker']
        for m in get('/trade-api/v2/events/' + et)['markets']:
            result = m.get('result')
            if result not in ('yes', 'no'):
                continue
            trades = fetch_trades(m['ticker'])
            if len(trades) < 5:
                continue
            n_band = 0
            for (t, size, side, yp, price_paid) in trades:
                if not (BAND[0] <= yp <= BAND[1]):
                    continue
                n_band += 1
                dir_sign = 1 if side == 'yes' else -1        # effect on YES price
                row = {'event': et, 'ticker': m['ticker'], 't': t, 'size': size,
                       'side': side, 'yes_price': yp, 'price_paid': price_paid,
                       'notional': size * price_paid, 'result': result}
                # settlement PnL following the aggressor's side
                payoff = 1.0 if side == result else 0.0
                row['win'] = int(payoff == 1.0)
                row['pnl'] = payoff - price_paid - taker_fee(price_paid)
                # short-horizon follow-through of the YES price
                for h in HORIZONS:
                    fyp = yes_price_at(trades, t + h)
                    row[f'ft_{h}'] = (fyp - yp) * dir_sign if fyp is not None else None
                records.append(row)
            match_summ.append((m['ticker'], result, len(trades), n_band))
            time.sleep(0.15)
    return pd.DataFrame(records), match_summ


if REFETCH or not os.path.exists(CACHE_CSV):
    df, match_summ = build_records()
    df.to_csv(CACHE_CSV, index=False)
    print('fetched & cached', len(df), 'trades from', len(match_summ), 'markets')
    for tk, res, ntr, nb in match_summ:
        print(f'  {tk:<44} result={res}  trades={ntr:<5} in-band={nb}')
else:
    df = pd.read_csv(CACHE_CSV)
    print('loaded', len(df), 'cached trades from', df['ticker'].nunique(), 'markets  (REFETCH=True to re-pull)')

df.head()

fetched & cached 509648 trades from 198 markets
  KXATPCHALLENGERMATCH-26JUL13KENSHE-KEN       result=no  trades=23464 in-band=22162
  KXATPCHALLENGERMATCH-26JUL13KENSHE-SHE       result=yes  trades=20542 in-band=19075
  KXATPCHALLENGERMATCH-26JUL13WONTUN-WON       result=yes  trades=2632  in-band=2180
  KXATPCHALLENGERMATCH-26JUL13WONTUN-TUN       result=no  trades=1842  in-band=1554
  KXATPCHALLENGERMATCH-26JUL13SANLOP-SAN       result=yes  trades=3623  in-band=2705
  KXATPCHALLENGERMATCH-26JUL13SANLOP-LOP       result=no  trades=6029  in-band=4973
  KXATPCHALLENGERMATCH-26JUL13RODALK-ROD       result=yes  trades=9403  in-band=8797
  KXATPCHALLENGERMATCH-26JUL13RODALK-ALK       result=no  trades=11369 in-band=11010
  KXATPCHALLENGERMATCH-26JUL13INCSTE-INC       result=yes  trades=15033 in-band=14068
  KXATPCHALLENGERMATCH-26JUL13INCSTE-STE       result=no  trades=18894 in-band=18269
  KXATPCHALLENGERMATCH-26JUL13PERTOB-PER       result=yes  trades=6523  in-band=6050
  KXATPCHALLENGER

,event,ticker,t,size,side,yes_price,price_paid,notional,result,win,pnl,ft_60,ft_300,ft_900
0,KXATPCHALLENGERMATCH-26JUL13KENSHE,KXATPCHALLENGERMATCH-26JUL13KENSHE-KEN,1.783951e+09,51.00,yes,0.56,0.56,28.5600,no,0,-0.577248,0.0,0.0,0.0
1,KXATPCHALLENGERMATCH-26JUL13KENSHE,KXATPCHALLENGERMATCH-26JUL13KENSHE-KEN,1.783953e+09,0.08,yes,0.54,0.54,0.0432,no,0,-0.557388,0.0,0.0,0.0
2,KXATPCHALLENGERMATCH-26JUL13KENSHE,KXATPCHALLENGERMATCH-26JUL13KENSHE-KEN,1.783954e+09,124.32,yes,0.54,0.54,67.1328,no,0,-0.557388,0.0,0.0,0.0
3,KXATPCHALLENGERMATCH-26JUL13KENSHE,KXATPCHALLENGERMATCH-26JUL13KENSHE-KEN,1.783955e+09,35.00,yes,0.54,0.54,18.9000,no,0,-0.557388,0.0,0.0,0.0
4,KXATPCHALLENGERMATCH-26JUL13KENSHE,KXATPCHALLENGERMATCH-26JUL13KENSHE-KEN,1.783955e+09,87.00,yes,0.54,0.54,46.9800,no,0,-0.557388,0.0,0.0,0.0


## 1. Summary by threshold

Raw, pooled view. **Watch `avg_price` climb with the threshold** — that's the favorite confound that Section 2 controls for. Take the rising win% here with a grain of salt.

In [9]:
def summary_by_threshold(df, thresholds=THRESHOLDS, horizons=HORIZONS):
    rows = []
    for th in thresholds:
        d = df[df['notional'] >= th]
        if len(d) == 0:
            continue
        rec = {'threshold': th, 'N': len(d), 'avg_size': d['size'].mean(),
               'avg_price': d['price_paid'].mean(), 'win%': 100 * d['win'].mean(),
               'pnl_per_ct': d['pnl'].mean(),
               'pnl_size_wtd': (d['pnl'] * d['size']).sum() / d['size'].sum()}
        for h in horizons:
            col = f'ft_{h}'
            v = d[col].dropna()
            rec[f'ft{h//60}m_%with'] = 100 * (v > 0).mean()
            rec[f'ft{h//60}m_meanc'] = 100 * v.mean()   # cents
        rows.append(rec)
    return pd.DataFrame(rows).set_index('threshold')


summary_by_threshold(df).round(3)

,N,avg_size,avg_price,win%,pnl_per_ct,pnl_size_wtd,ft1m_%with,ft1m_meanc,ft5m_%with,ft5m_meanc,ft15m_%with,ft15m_meanc
threshold,,,,,,,,,,,,
0,509648,148.782,0.501,49.272,-0.022,-0.020,41.127,-0.178,45.984,-0.129,48.071,-0.277
100,59269,907.280,0.614,61.990,-0.007,-0.015,42.327,-0.076,48.535,-0.034,53.713,-0.035
250,30659,1384.301,0.651,65.925,-0.004,-0.014,43.684,0.019,50.106,0.044,55.700,0.054
500,15159,2046.414,0.684,69.530,-0.001,-0.012,44.086,0.028,50.854,0.105,56.633,-0.105
1000,5917,3228.022,0.733,73.669,-0.008,-0.018,42.369,-0.128,50.701,-0.239,58.611,-0.395
1500,3317,4256.440,0.744,74.405,-0.011,-0.021,42.267,-0.137,50.377,-0.459,58.577,-0.710
2000,1987,5311.448,0.760,74.987,-0.021,-0.029,41.419,-0.348,50.025,-0.772,60.141,-0.732


## 2. Big vs. all, within price buckets  (the real test)

If *size* carries information beyond the price, BIG should beat ALL **inside** a price bucket. In the pilot, favorite buckets (≥0.50) showed big ≈ all — no edge over the price. The only cell where big out-performed was the **underdog** side (price < 0.50), consistent with informed money backing a mispriced longshot. Watch the `n` — underdog cells thin out fast at high thresholds.

In [10]:
def bucket_compare(df, thresholds=(0, 500, 1000), buckets=PRICE_BUCKETS, metric='pnl'):
    """metric: 'pnl' (mean PnL/contract) or 'win' (win%). One column per threshold (0 = ALL)."""
    out = {}
    for th in thresholds:
        col = {}
        for lo, hi in buckets:
            d = df[(df['price_paid'] >= lo) & (df['price_paid'] < hi) & (df['notional'] >= th)]
            if metric == 'pnl':
                val = d['pnl'].mean()
            else:
                val = 100 * d['win'].mean()
            col[f'{lo:.2f}-{hi:.2f}'] = f'{val:+.3f} (n={len(d)})' if metric=='pnl' else f'{val:.1f}% (n={len(d)})'
        out['ALL' if th == 0 else f'>=${th}'] = col
    return pd.DataFrame(out)


print('PnL per contract (net of fee):')
display(bucket_compare(df, thresholds=THRESHOLDS, metric='pnl'))
print('\nWin %:')
display(bucket_compare(df, thresholds=THRESHOLDS, metric='win'))

PnL per contract (net of fee):


,ALL,>=$100,>=$250,>=$500,>=$1000,>=$1500,>=$2000
0.05-0.35,-0.047 (n=157095),-0.030 (n=8635),-0.025 (n=3061),-0.027 (n=946),-0.021 (n=205),-0.018 (n=94),-0.052 (n=45)
0.35-0.50,-0.033 (n=89788),-0.018 (n=9079),-0.014 (n=4173),-0.020 (n=1794),-0.008 (n=546),+0.057 (n=227),+0.029 (n=145)
0.50-0.65,-0.018 (n=98238),-0.022 (n=13380),-0.024 (n=7062),-0.016 (n=3477),-0.043 (n=1093),-0.072 (n=616),-0.117 (n=266)
0.65-0.80,-0.001 (n=80189),-0.002 (n=11707),-0.000 (n=6435),+0.004 (n=3290),-0.013 (n=1186),-0.024 (n=747),-0.039 (n=459)
0.80-0.95,+0.016 (n=78213),+0.027 (n=15042),+0.026 (n=9092),+0.026 (n=5143),+0.022 (n=2590),+0.025 (n=1471),+0.025 (n=971)



Win %:


,ALL,>=$100,>=$250,>=$500,>=$1000,>=$1500,>=$2000
0.05-0.35,15.5% (n=157095),20.2% (n=8635),22.2% (n=3061),22.5% (n=946),23.9% (n=205),25.5% (n=94),22.2% (n=45)
0.35-0.50,40.6% (n=89788),42.5% (n=9079),43.0% (n=4173),43.1% (n=1794),44.1% (n=546),50.7% (n=227),48.3% (n=145)
0.50-0.65,56.8% (n=98238),56.5% (n=13380),56.5% (n=7062),57.6% (n=3477),55.5% (n=1093),52.8% (n=616),48.5% (n=266)
0.65-0.80,73.2% (n=80189),73.3% (n=11707),73.6% (n=6435),73.8% (n=3290),72.2% (n=1186),71.4% (n=747),69.9% (n=459)
0.80-0.95,89.5% (n=78213),91.0% (n=15042),91.0% (n=9092),91.1% (n=5143),91.0% (n=2590),91.0% (n=1471),91.0% (n=971)


## 3. Per-match consistency

Pooled stats can hide a few matches driving everything. This checks how many individual markets are net-positive when following big trades to settlement.

In [11]:
def per_match(df, threshold=500):
    d = df[df['notional'] >= threshold]
    g = d.groupby('ticker').agg(n_big=('pnl', 'size'), total_pnl=('pnl', 'sum'),
                                win_rate=('win', 'mean'), avg_price=('price_paid', 'mean'))
    g['win_rate'] = (100 * g['win_rate']).round(1)
    pos = (g['total_pnl'] > 0).sum()
    print(f'threshold >= ${threshold}:  {len(g)} markets with a big trade, '
          f'{pos} net-positive to settlement ({100*pos/len(g):.0f}%)')
    return g.sort_values('total_pnl', ascending=False).round(3)


per_match(df, threshold=500)

threshold >= $500:  190 markets with a big trade, 124 net-positive to settlement (65%)


,n_big,total_pnl,win_rate,avg_price
ticker,,,,
KXATPCHALLENGERMATCH-26JUL13NOGBRO-NOG,558,115.972,91.4,0.694
KXATPCHALLENGERMATCH-26JUL13KENSHE-SHE,845,88.841,80.8,0.691
KXATPCHALLENGERMATCH-26JUL13PERTOB-PER,320,77.253,94.4,0.689
KXATPCHALLENGERMATCH-26JUL13AGABAR-BAR,413,71.339,83.1,0.644
KXATPCHALLENGERMATCH-26JUL13RODALK-ROD,322,67.963,96.0,0.737
...,...,...,...,...
KXATPCHALLENGERMATCH-26JUL12GEOMAG-MAG,153,-69.150,22.2,0.659
KXATPCHALLENGERMATCH-26JUL13POLMIY-POL,418,-87.584,50.5,0.702
KXATPCHALLENGERMATCH-26JUL12FEAWAL-WAL,566,-168.767,43.5,0.722


## Next steps (not yet built)

- **Realistic entry:** replace `price_paid` (the big trade's own fill) with the *post-trade ask* so PnL reflects paying up. The 1-min follow-through being <50% means chasing fills worse — this is the make-or-break correction.
- **Scale the sample:** raise `N_EVENTS` / loop over several days so the underdog (price<0.35) big-trade cell has thousands of obs, not ~200.
- **Size-matched control:** compare big vs. a random same-price, same-time-in-match trade, not just all trades.